# 06. Final training with 3-fold CV, inference and saved results

**What this notebook does:** trains the final model (all decisions from notebooks 00-05 in `nerkit/decisions.py`) once per frozen CV fold (0, 1, 2), scores each held-out fold, predicts the competition test set with each fold model, and saves everything so it can be reused: logits for every split, tokenizer, model weights, scores and one Kaggle-format CSV per fold.

**Settings:** GRU + CRF, word tokens with frozen fastText aligned vectors, 100k rows per language, structure tokens, early stopping on a 5% inner validation split, seed 3012026. Decoding: lenient merge, empty strings dropped, CRF bias 0.

**Where to run:** Colab with a GPU (Runtime > Change runtime type > GPU). It also runs locally; then it just takes longer. A finished fold is skipped on rerun, so a disconnected session resumes where it stopped.

**Before the first run on Colab**, put these into your Google Drive:
- the project folder `ai301/deep-learning-assignment3` (cloned automatically from the `model` branch on GitHub if missing; `model/` has to be pushed there first), and
- the four Kaggle files (`uk_geo_dataset.csv`, `ru_geo_dataset.csv`, `test.csv`, `labeling_sample.csv`) in `MyDrive/ai301/data/kse-ua-location-extraction-2026/`.

Everything the notebook writes goes to `MyDrive/ai301/ner_work/` on Colab, so it survives a disconnect.

In [4]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive/ai301")
    ROOT = PROJECT / "deep-learning-assignment3"
    BRANCH = "model"  # the branch that holds model/; main does not have it
    if not ROOT.exists():
        PROJECT.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "-b", BRANCH, "https://github.com/koalla05/deep-learning-assignment3.git", str(ROOT)], check=True)
    else:
        subprocess.run(["git", "fetch", "origin", BRANCH], cwd=ROOT, check=True)
        subprocess.run(["git", "checkout", BRANCH], cwd=ROOT, check=True)
        subprocess.run(["git", "pull", "origin", BRANCH], cwd=ROOT, check=True)
    # The frozen split manifests are stored with Git LFS; a plain clone only has small pointer files.
    subprocess.run("apt-get install -y -q git-lfs", shell=True)
    subprocess.run(["git", "lfs", "install"], cwd=ROOT)
    subprocess.run(["git", "lfs", "pull"], cwd=ROOT)
    import urllib.request

    for name in ("role_manifest_v2.parquet", "cv_manifest_v1.parquet"):
        manifest = ROOT / "eda" / "output" / name
        if manifest.read_bytes()[:4] != b"PAR1":  # still a pointer file: download the real one
            url = f"https://media.githubusercontent.com/media/koalla05/deep-learning-assignment3/{BRANCH}/eda/output/{name}"
            urllib.request.urlretrieve(url, manifest)
        assert manifest.read_bytes()[:4] == b"PAR1", f"{name} is not a parquet file; download it from GitHub and upload it to {manifest.parent}"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "lightning", "pytorch-crf", "tokenizers"], check=True)
    os.environ["NER_DATA_DIR"] = str(PROJECT / "data" / "kse-ua-location-extraction-2026")
    os.environ["NER_WORK_DIR"] = str(PROJECT / "ner_work")
else:
    ROOT = Path.cwd()
    while not (ROOT / "eda").exists():
        ROOT = ROOT.parent
    # Results downloaded from Colab (folder ner_work) are continued here: finished folds are skipped.
    if "NER_WORK_DIR" not in os.environ and (ROOT / "data" / "ner_work").exists():
        os.environ["NER_WORK_DIR"] = str(ROOT / "data" / "ner_work")

import torch

os.environ.setdefault("NER_WANDB", "off")
os.environ["NER_DEVICE"] = "cuda" if torch.cuda.is_available() else "cpu"
sys.path[:0] = [str(ROOT), str(ROOT / "model")]
print("Device:", os.environ["NER_DEVICE"], "| project:", ROOT, "| work folder:", os.environ.get("NER_WORK_DIR", ROOT / "model"))

Device: cpu | project: /Users/allakovalcuk/AI301/deep-learning-assignment3 | work folder: /Users/allakovalcuk/AI301/deep-learning-assignment3/data/ner_work


In [5]:
import json
import shutil

import pandas as pd
from IPython.display import display

from nerkit.artifacts import combine_models, decode_split, load_logits, write_submission
from nerkit.config import ARTIFACTS_DIR, WORK_DIR
from nerkit.data import build_cache, load_test
from nerkit.decisions import DECISIONS, DECODING, base_config
from nerkit.experiment import read_results, run_experiment, select_training
from nerkit.metrics import evaluate

NOTEBOOK = "06_final_cv_training"
FOLDS = (0, 1, 2)
DECODE = {name: DECODING[name] for name in ("merge", "drop_empty", "trim_number", "bias")}

build_cache()  # joins the CSV files with the frozen manifests; skipped when already cached
print("Work folder:", WORK_DIR)
print(json.dumps(DECISIONS, indent=1))

Work folder: /Users/allakovalcuk/AI301/deep-learning-assignment3/data/ner_work
{
 "tokenizer": "word",
 "vocab_size": 16000,
 "lowercase": true,
 "case_feature": true,
 "pretrained": "aligned",
 "embedding_mode": "frozen",
 "pretrained_top_k": 200000,
 "max_epochs": 10,
 "patience": 3,
 "rnn": "gru",
 "crf": true,
 "optimizer": "adamw",
 "lr": 0.003,
 "weight_decay": 0.01,
 "scheduler": "none",
 "batch_size": 32,
 "dropout": 0.15,
 "grad_clip": 1.0,
 "uk_rows": 100000,
 "ru_rows": 100000,
 "structure_tokens": true
}


## Train one model per fold

Fold `k` trains on the other two folds (plus the `extra` training rows) and is scored on fold `k`, which the model never sees. The 5% inner split is used only for early stopping.

In [6]:
records = [run_experiment(f"fold{fold}", base_config(fold=fold), NOTEBOOK) for fold in FOLDS]

fold0: already finished, F1 mix 0.885
fold1: already finished, F1 mix 0.896
fold2: already finished, F1 mix 0.894


## Cross-validation scores

Re-decoded from the saved held-out logits with the final decoding settings, so the scores can be reproduced from the files alone.

In [7]:
rows_by_fold, scores = {}, []
for record in records:
    fold = record["config"]["fold"]
    _, _, _, heldout = select_training(base_config(fold=fold))
    logits, meta = load_logits(record["key"], "heldout")
    result = evaluate(heldout, decode_split(logits, heldout, crf=meta["crf"], **DECODE))
    scores.append(dict(fold=fold, mix_f1=result["mix"]["f1"], precision=result["mix"]["precision"],
                       recall=result["mix"]["recall"], uk_f1=result["uk"]["f1"], ru_f1=result["ru"]["f1"],
                       posts_f1=record["scores"]["posts"]["mix"]["f1"],
                       epochs=len(record["history"]), minutes=record["total_seconds"] / 60))
table = pd.DataFrame(scores).set_index("fold")
display(table.round(3))
cv = pd.DataFrame({"mean": table.mean(), "std": table.std()}).round(3)
display(cv)

,mix_f1,precision,recall,uk_f1,ru_f1,posts_f1,epochs,minutes
fold,,,,,,,,
0,0.885,0.914,0.858,0.875,0.953,0.863,6,39.079
1,0.896,0.912,0.880,0.889,0.942,0.873,10,62.448
2,0.894,0.906,0.881,0.886,0.947,0.882,6,10.514


,mean,std
mix_f1,0.891,0.006
precision,0.911,0.004
recall,0.873,0.013
uk_f1,0.883,0.008
ru_f1,0.947,0.005
posts_f1,0.873,0.009
epochs,7.333,2.309
minutes,37.347,26.011


## Test predictions

Every fold model predicts the 477 competition posts. One CSV per fold is written, so they can be submitted separately; combining them is left to the ensembling step. The logits of all splits are already saved next to each model.

In [8]:
test_rows = load_test()
submissions = {}
for record in records:
    fold = record["config"]["fold"]
    logits, meta = load_logits(record["key"], "test")
    predictions = decode_split(logits, test_rows, crf=meta["crf"], **DECODE)
    path = write_submission(WORK_DIR / "submissions" / f"final_fold{fold}.csv", test_rows, predictions)
    submissions[fold] = path
    print(f"fold {fold}: {sum(bool(p) for p in predictions)} of {len(predictions)} posts have locations, "
          f"{sum(len(p) for p in predictions)} locations in total -> {path.name}")

fold 0: 317 of 477 posts have locations, 701 locations in total -> final_fold0.csv
fold 1: 333 of 477 posts have locations, 749 locations in total -> final_fold1.csv
fold 2: 326 of 477 posts have locations, 758 locations in total -> final_fold2.csv


## Leaderboard scores of the three fold submissions

Each fold CSV was uploaded to Kaggle once (public score, F1 on the 477 test posts). The local CV score is measured on sentences taken from the training data; the leaderboard measures competition posts, which look different. The drop from about 0.89 to about 0.58 is therefore expected to be mostly a difference in data and labeling, not a bug in the models. For comparison, Herman's earlier submission scored 0.411.

In [12]:
LEADERBOARD = {0: 0.57098, 1: 0.58308, 2: 0.58594}  # public score of final_fold<k>.csv, copied from Kaggle

comparison = table[["mix_f1"]].rename(columns={"mix_f1": "local_cv_f1"})
comparison["leaderboard_f1"] = pd.Series(LEADERBOARD)
comparison["gap"] = comparison["local_cv_f1"] - comparison["leaderboard_f1"]
display(comparison.round(3))
print(f"mean local CV {comparison['local_cv_f1'].mean():.3f}, mean leaderboard {comparison['leaderboard_f1'].mean():.3f}, "
      f"leaderboard spread (std) {comparison['leaderboard_f1'].std():.3f}")

,local_cv_f1,leaderboard_f1,gap
fold,,,
0,0.885,0.571,0.314
1,0.896,0.583,0.313
2,0.894,0.586,0.308


mean local CV 0.891, mean leaderboard 0.580, leaderboard spread (std) 0.008


## Everything saved for reuse

`final_cv/` collects what is needed to ensemble or to run the models again: per fold the logits for `inner`, `heldout`, `posts`, `labeling` and `test` (float16, one row per token, order O/B/I, CRF transition scores in the metadata), the tokenizer, the model weights, plus `summary.json` with the configuration, the CV scores and the decoding settings. `final_cv.zip` is the same folder for downloading.

In [9]:
package = WORK_DIR / "final_cv"
package.mkdir(parents=True, exist_ok=True)
for record in records:
    fold = record["config"]["fold"]
    shutil.copytree(ARTIFACTS_DIR / record["key"], package / f"fold{fold}", dirs_exist_ok=True)
summary = dict(
    decisions=DECISIONS, decoding=DECODE, seed=records[0]["config"]["seed"],
    folds={f"fold{r['config']['fold']}": dict(run_key=r["key"], scores=r["scores"], epochs=len(r["history"]),
                                              training_rows=r["training_rows"], minutes=r["total_seconds"] / 60)
           for r in records},
    cv_mean=table.mean().to_dict(), cv_std=table.std().to_dict(),
    leaderboard={f"fold{k}": v for k, v in LEADERBOARD.items()}, tag_order=["O", "B", "I"], note="Logits per split are in fold<k>/<split>.npz; see notebook 05 for the format.")
(package / "summary.json").write_text(json.dumps(summary, indent=1, ensure_ascii=False), encoding="utf-8")
shutil.copytree(WORK_DIR / "submissions", package / "submissions", dirs_exist_ok=True)
archive = shutil.make_archive(str(WORK_DIR / "final_cv"), "zip", package)
print("Saved:", package)
print("Archive:", archive, f"({Path(archive).stat().st_size / 1e6:.0f} MB)")
if IN_COLAB:
    from google.colab import files

    files.download(archive)

Saved: /Users/allakovalcuk/AI301/deep-learning-assignment3/data/ner_work/final_cv
Archive: /Users/allakovalcuk/AI301/deep-learning-assignment3/data/ner_work/final_cv.zip (24 MB)


## Log the finished runs to Weights & Biases (optional)

Logs one W&B run per fold after training: the per-epoch curves (training and validation loss, validation F1, precision, recall, learning rate) and the final scores, plus a run with the CV mean. Nothing is retrained. Set `USE_WANDB = False` to skip it. The first time, `wandb.login()` asks for your API key from wandb.ai/authorize.

In [10]:
import getpass, os
os.environ["WANDB_BASE_URL"] = "https://api.forge.coreweave.com"
os.environ["WANDB_API_KEY"] = getpass.getpass("W&B API key: ")

In [11]:
USE_WANDB = True
WANDB_PROJECT = "ai301-ner-final"

if USE_WANDB:
    try:
        import wandb
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
        import wandb
    if os.environ.get("WANDB_MODE") != "offline":
        wandb.login()
    for record in records:
        fold = record["config"]["fold"]
        run = wandb.init(project=WANDB_PROJECT, group="06_final_cv", name=f"fold{fold}", config=record["config"])
        for entry in record["history"]:
            run.log({name: value for name, value in entry.items() if name != "epoch"}, step=entry["epoch"])
        held = record["scores"]["heldout"]
        run.summary.update({
            "heldout_f1_mix": held["mix"]["f1"], "heldout_precision": held["mix"]["precision"],
            "heldout_recall": held["mix"]["recall"], "heldout_f1_uk": held["uk"]["f1"],
            "heldout_f1_ru": held["ru"]["f1"], "posts_f1": record["scores"]["posts"]["mix"]["f1"],
            "labeling_f1": record["scores"]["labeling"]["pooled"]["f1"],
            "uk_unseen_recall": record["scores"]["unseen_recall"].get("uk"),
            "epochs": len(record["history"]), "train_seconds": record["train_seconds"],
            "device": record["device"], "run_key": record["key"]})
        run.finish()
    summary_run = wandb.init(project=WANDB_PROJECT, group="06_final_cv", name="cv_mean",
                             config=dict(decisions=DECISIONS, decoding=DECODE))
    summary_run.summary.update({f"cv_mean_{name}": value for name, value in table.mean().items()}
                               | {f"cv_std_{name}": value for name, value in table.std().items()})
    summary_run.finish()


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


lr,▁▁▁▁▁▁
seconds,▂▁█▇▃▃
train_loss,█▃▂▂▁▁
val_f1,▁▄█▇▆▆
val_loss,█▄▂▄▁▅
val_precision,▁▇█▄▆▅
val_recall,▅▁▄█▄▆
device,cuda
epochs,6
heldout_f1_mix,0.88484
heldout_f1_ru,0.95291


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


lr,▁▁▁▁▁▁▁▁▁▁
seconds,▁▂▃▄▃▄▅█▅▅
train_loss,█▄▃▂▂▂▁▁▁▁
val_f1,▁▃▅▆▅▅▇▅▅█
val_loss,█▂▂▁▁▂▂▂▂▂
val_precision,▁▄▄▇█▆▄▅▅▇
val_recall,█▄▇▃▁▅█▆▆▆
device,cuda
epochs,10
heldout_f1_mix,0.89597
heldout_f1_ru,0.94209


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


lr,▁▁▁▁▁▁
seconds,▇█▆▄▂▁
train_loss,█▃▂▂▁▁
val_f1,▁▅█▇▅▃
val_loss,█▇▄▃▄▁
val_precision,▆█▂▆▆▁
val_recall,▁▃█▆▄▄
device,cpu
epochs,6
heldout_f1_mix,0.89356
heldout_f1_ru,0.94708


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


cv_mean_epochs,7.33333
cv_mean_minutes,37.34686
cv_mean_mix_f1,0.89146
cv_mean_posts_f1,0.87269
cv_mean_precision,0.91079
cv_mean_recall,0.87303
cv_mean_ru_f1,0.94736
cv_mean_uk_f1,0.8831
cv_std_epochs,2.3094
cv_std_minutes,26.01054
+6,...
